In [ ]:
class Particle:
    def __init__(self, num_dimensions):
        self.position = np.random.uniform(0, 1, size=(num_dimensions,))  # Initialize position randomly
        self.velocity = np.random.uniform(-1, 1, size=(num_dimensions,))  # Initialize velocity randomly
        self.personal_best_position = self.position.copy()  # Personal best position
        self.personal_best_fitness = float('inf')  # Personal best fitness

    def evaluate_fitness(self, objective_function):
        self.fitness = objective_function(self.position)  # Calculate current fitness

        if self.fitness < self.personal_best_fitness:  # Update personal best
            self.personal_best_position = self.position.copy()
            self.personal_best_fitness = self.fitness

    def update_velocity(self, global_best_position, c1, c2):
        r1 = random.random()
        r2 = random.random()

        cognitive_velocity = c1 * r1 * (self.personal_best_position - self.position)
        social_velocity = c2 * r2 * (global_best_position - self.position)
        self.velocity += cognitive_velocity + social_velocity

    def update_position(self, bounds):
        for i in range(len(self.position)):
            self.position[i] += self.velocity[i]  # Update position based on velocity

            # Adjust position if necessary to stay within bounds
            if self.position[i] < bounds[i][0]:
                self.position[i] = bounds[i][0]
            elif self.position[i] > bounds[i][1]:
                self.position[i] = bounds[i][1]


In [ ]:
import numpy as np
import pandas as pd

import matplotlib
import matplotlib.mlab as ml
import matplotlib.pyplot as plt

from numpy.random import randn, shuffle
from scipy import linspace, meshgrid, arange, empty, concatenate, newaxis, shape

from matplotlib.ticker import MaxNLocator
from matplotlib import cm
from mpl_toolkits.mplot3d import Axes3D

In [ ]:
# Load the dataset

data = pd.read_csv('data_file.csv', names=['x', 'y'])
x_cars = data['x'].to_numpy()
y_cars = data['y'].to_numpy()

In [ ]:
# Define the basestation coordinates

x_base1 = 0
y_base1 = 0
x_base2 = 1000
y_base2 = 1000

In [ ]:
# Define the minimum transmission rate that must be satisfied for each car. T

min_transmission_rate = 1

In [ ]:
# Initialize the PSO algorithm parameters

num_particles = 100
num_iterations = 200
inertia_weight = 0.7298 #(between 0.9 - 0.4)

In [ ]:
# Create the PSO swarm; each representing a potential solution to the transmission rate optimization problem.
# Initialize the position and velocity of each particle.

swarm = []
for _ in range(num_particles):
    particle = Particle(num_particles)
    swarm.append(particle)

In [ ]:
# Define the Objective Function that we want to minimize

def objective_function(transmission_rates):
    total_transmission_rate = 0
    for i in range(len(x_cars)):
        for j in range(len(transmission_rates)):
            distance = np.sqrt((x_cars[i] - x_base1[j])**2 + (y_cars[i] - y_base1[j])**2)
            transmission_rate = transmission_rates[j]
            total_transmission_rate += 1 / distance * transmission_rate

    return total_transmission_rate

In [ ]:
# Implement PSO Update

# Update Velocity
def update_velocity(particle, personal_best_position, global_best_position):
    c1 = 1.4962
    c2 = 1.4962
    r1 = random.random()
    r2 = random.random()

    cognitive_velocity = c1 * r1 * (personal_best_position - particle.position)
    social_velocity = c2 * r2 * (global_best_position - particle.position)
    particle.velocity = inertia_weight * particle.velocity + cognitive_velocity + social_velocity


# Update Position
def update_position(particle):
    particle.position += particle.velocity

    # Ensure that the transmission rates are greater than or equal to the minimum transmission rate
    for i in range(len(particle.position)):
        if particle.position[i] < min_transmission_rate:
            particle.position[i] = min_transmission_rate

In [ ]:
# Train the PSO with Update Iterations until convergence
for iteration in range(num_iterations):
    for particle in swarm:
        particle.evaluate_fitness(objective_function)
        particle.update_velocity(swarm[int(particle.personal_best_index)].position, swarm[global_best_particle_index].position)
        particle.update_position()

    global_best_particle_index = np.argmin([swarm[i].fitness for i in range(num_particles)])


In [ ]:
# Evaluate the results

best_transmission_rates = swarm[global_best_particle_index].position
total_transmission_rate = objective_function(best_transmission_rates)

print('Total transmission rate:', total_transmission_rate)

The problem is to find the shortest rate of transmission between two basestations and through 100 cars on the highway. The transmission rate (tr) is inversely proportional to the distance (d) between the basestations and the cars, and the distance is given by the formula

The objective function is to minimize the total transmission rate between the basestations and the cars. The total transmission rate is the sum of the transmission rates between each car and each base station. The transmission rate between a car and a base station is given by

The constraints are that the transmission rate between each car and each base station must be greater than or equal to a minimum transmission rate. The minimum transmission rate is necessary to ensure that the data is transmitted reliably.